# calibrate.ipynb — Heston DDN 模型校准（IV 空间 + Feller 约束版）

本 Notebook 负责：
1. 加载已训练的 `HestonDDN` 权重（IV 目标版本）
2. 构建单参数期权链 → 计算市场 IV 和 Vega
3. 运行 Multi-start Adam 校准: **Vega 加权 IV MSE + Feller 惩罚**
4. 对比校准值 vs Ground Truth
5. 用校准后参数重新定价验证

**核心升级**:
- 校准目标: $\mathcal{L} = \frac{1}{M}\sum \text{Vega}_i \cdot (\hat{IV}_i - IV_i^{\text{mkt}})^2 + \lambda_F \cdot \text{ReLU}(\sigma^2 - 2\kappa\theta + \epsilon)$
- Feller 条件 $2\kappa\theta > \sigma^2$ 作为物理约束（PINN 思想）

In [1]:
## Section 8: 导入依赖与项目路径配置
import sys
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd()   # 运行时当前目录 = Heston_Project/
sys.path.insert(0, str(PROJECT_ROOT))

from modules.model       import HestonDDN
from modules.calibration import HestonCalibrator
from modules.pricing     import calculate_heston_price, compute_iv_vega_batch, check_feller

import numpy as np
import pandas as pd
import torch

# ---- 路径常量 ----
MODEL_PATH = PROJECT_ROOT / "models" / "heston_ddn_weights.pth"

# 兼容：若本项目 models/ 下无权重，则尝试上级目录
FALLBACK_MODEL = Path("/Users/liaojiansong/calibration/heston_ddn_weights.pth")

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"MODEL_PATH   : {MODEL_PATH}")

PROJECT_ROOT : /Users/liaojiansong/calibration/Heston_Project
MODEL_PATH   : /Users/liaojiansong/calibration/Heston_Project/models/heston_ddn_weights.pth


In [2]:
## Section 9: 加载已训练模型权重（IV 目标版本）

device = torch.device(
    "mps"  if torch.backends.mps.is_available() else
    "cuda" if torch.cuda.is_available() else "cpu"
)
print(f"使用设备: {device}")

# 确定权重文件路径
if MODEL_PATH.exists():
    ckpt_path = MODEL_PATH
elif FALLBACK_MODEL.exists():
    ckpt_path = FALLBACK_MODEL
    print(f"⚠️  使用备用权重: {ckpt_path}")
else:
    raise FileNotFoundError(
        f"未找到模型权重。请先运行 01_train.ipynb，或将权重放置于: {MODEL_PATH}"
    )

# 加载
ckpt  = torch.load(ckpt_path, map_location=device)
model = HestonDDN(
    input_dim   = ckpt.get('input_dim',  8),
    heston_param_dim = ckpt.get('heston_dim', 5),
).to(device)
model.load_state_dict(ckpt['model_state_dict'])
model.is_fitted = ckpt.get('is_fitted', True)
model.eval()

target_type = ckpt.get('target', 'price_norm')  # 兼容旧版
print(f"✅ 权重已加载: {ckpt_path}")
print(f"   模型目标  = {target_type}")
print(f"   input_dim = {model.input_dim}")
print(f"   is_fitted = {model.is_fitted}")
print(f"   y_min = {model.y_min.item():.6f}")
print(f"   y_max = {model.y_max.item():.6f}")
print(f"   X_min = {model.X_min.cpu().numpy()}")
print(f"   X_max = {model.X_max.cpu().numpy()}")

使用设备: mps
✅ 权重已加载: /Users/liaojiansong/calibration/Heston_Project/models/heston_ddn_weights.pth
   模型目标  = iv
   input_dim = 8
   is_fitted = True
   y_min = 0.093227
   y_max = 1.038616
   X_min = [ 0.01369767  0.01011868  0.10000127 -0.8999977   0.01000155  0.00100037
  0.05000845 -0.49998844]
   X_max = [ 4.9999504   0.99999684  0.99998355 -0.05000427  0.99999714  0.09999973
  0.9999982   0.49997777]


## 市场数据模拟校准

In [15]:
## Section 10: 构建单参数期权链 + 计算市场 IV & Vega

# ---- Ground Truth Heston 参数 ----
true_kappa  = 2.0
true_lambda = 0.1
true_sigma  = 0.4
true_rho    = -0.6
true_v0     = 0.05
true_r      = 0.05
true_S0     = 1000.0

true_heston = np.array([true_kappa, true_lambda, true_sigma, true_rho, true_v0])

# ── Feller 条件检查 ──
feller_val = 2 * true_kappa * true_lambda
print("Ground Truth Heston 参数:")
for n, v in zip(['kappa','lambda','sigma','rho','v0'], true_heston):
    print(f"  {n:>8s} = {v:.4f}")
print(f"  {'r':>8s} = {true_r}")
print(f"  {'S0':>8s} = {true_S0}")
print(f"\n  Feller 检查: 2κθ = {feller_val:.4f} vs σ² = {true_sigma**2:.4f}"
      f" → {'满足 ✓' if check_feller(true_kappa, true_lambda, true_sigma) else '不满足 ✗'}")
print()

# ---- 构建期权链网格 ----
taus      = [0.3, 0.4, 0.5, 0.6, 0.7]
moneyness = np.linspace(0.6, 1.5, 20)

rows = []
for tau in taus:
    for m in moneyness:
        K    = true_S0 * m
        Pmkt = calculate_heston_price(
            true_kappa, true_lambda, true_sigma, true_rho, true_v0,
            true_r, tau, true_S0, K
        )
        if not np.isnan(Pmkt) and Pmkt > 0.01:
            rows.append({'r': true_r, 'tau': tau, 'S0': true_S0, 'K': K, 'P_mkt': Pmkt})

sample_df = pd.DataFrame(rows)

# ── 计算市场 IV 和 Vega（核心升级: 校准目标改为 IV 空间） ──
iv_arr, vega_arr = compute_iv_vega_batch(
    sample_df['P_mkt'].values,
    sample_df['S0'].values,
    sample_df['K'].values,
    sample_df['tau'].values,
    sample_df['r'].values,
)
sample_df['iv_market'] = iv_arr
sample_df['vega']      = vega_arr

# 清洗 IV/Vega 无效的行
n_before = len(sample_df)
sample_df = sample_df.dropna(subset=['iv_market', 'vega'])
sample_df = sample_df[(sample_df['iv_market'] > 0) & (sample_df['vega'] > 0)].copy()
if len(sample_df) < n_before:
    print(f"⚠️  IV/Vega 清洗: {n_before} → {len(sample_df)} 条")

print(f"✅ 有效期权数: {len(sample_df)}")
print(sample_df[['tau', 'K', 'P_mkt', 'iv_market', 'vega']].to_string(index=False))

Ground Truth Heston 参数:
     kappa = 2.0000
    lambda = 0.1000
     sigma = 0.4000
       rho = -0.6000
        v0 = 0.0500
         r = 0.05
        S0 = 1000.0

  Feller 检查: 2κθ = 0.4000 vs σ² = 0.1600 → 满足 ✓

✅ 有效期权数: 99
 tau           K      P_mkt  iv_market     vega
 0.3  600.000000 408.976956   0.315900 0.016685
 0.3  647.368421 362.472633   0.316112 0.059732
 0.3  694.736842 316.206185   0.308479 0.145369
 0.3  742.105263 270.435318   0.298965 0.298857
 0.3  789.473684 225.604390   0.288921 0.543792
 0.3  836.842105 182.404284   0.278791 0.888667
 0.3  884.210526 141.798308   0.268761 1.309557
 0.3  931.578947 104.973464   0.258943 1.737877
 0.3  978.947368  73.180175   0.249452 2.066553
 0.3 1026.315789  47.454416   0.240436 2.184747
 0.3 1073.684211  28.281682   0.232099 2.034317
 0.3 1121.052632  15.347665   0.224683 1.654601
 0.3 1168.421053   7.558355   0.218423 1.171512
 0.3 1215.789474   3.394375   0.213470 0.725718
 0.3 1263.157895   1.407510   0.209833 0.398942
 0.3 13

In [16]:
## Section 11: Multi-start Adam 校准执行（IV 空间 + Feller 约束）
import time
# ── 构建 market_data 字典（包含 IV 和 Vega） ──
market_data = {
    'r':         sample_df['r'].values.astype(np.float32),
    'tau':       sample_df['tau'].values.astype(np.float32),
    'S0':        sample_df['S0'].values.astype(np.float32),
    'K':         sample_df['K'].values.astype(np.float32),
    'iv_market': sample_df['iv_market'].values,    # ★ 市场 IV
    'vega':      sample_df['vega'].values,          # ★ Vega 权重
}

calibrator = HestonCalibrator(model, device, seed=0)

t0 = time.time()
best_theta, best_loss, all_results = calibrator.calibrate(
    market_data,
    n_starts      = 10,
    lr            = 5e-3,
    max_steps     = 500,
    patience      = 40,
    lambda_feller = 10.0,     # ★ Feller 惩罚权重
    verbose       = True,
)

print(f"校准完成! 最佳参数: {best_theta}, 最佳损失: {best_loss:.6f}, 耗时: {time.time() - t0:.2f} 秒")


  市场期权数   : 99
  初始点数     : 10
  Adam lr      : 0.005，max_steps=500，patience=40
  Feller λ     : 10.0
  market IV    : [0.2052, 0.3349]

  [start  1/10] step= 100 | loss=2.9338e-02 | Feller=✓(0.00e+00) | lr=5.00e-03
  [start  1/10] step= 200 | loss=7.3024e-04 | Feller=✓(0.00e+00) | lr=5.00e-03
  [start  1/10] step= 300 | loss=5.5494e-04 | Feller=✓(0.00e+00) | lr=5.00e-03
  [start  1/10] step= 400 | loss=4.2400e-04 | Feller=✓(0.00e+00) | lr=5.00e-03
  [start  1/10] step= 500 | loss=3.2748e-04 | Feller=✓(0.00e+00) | lr=5.00e-03
  [start  1/10] loss=3.2748e-04  [κ=3.5883, λ=0.0441, σ=0.3512, ρ=-0.7500, v₀=0.1061] Feller✓ ★
  [start  2/10] step= 100 | loss=4.8118e-02 | Feller=✓(0.00e+00) | lr=5.00e-03
  [start  2/10] step= 200 | loss=8.0868e-03 | Feller=✓(0.00e+00) | lr=5.00e-03
  [start  2/10] step= 300 | loss=8.0488e-04 | Feller=✓(0.00e+00) | lr=5.00e-03
  [start  2/10] step= 400 | loss=5.2350e-04 | Feller=✓(0.00e+00) | lr=5.00e-03
  [start  2/10] step= 500 | loss=3.8629e-04 | Feller=✓(0

In [17]:
## Section 12: 校准结果 vs Ground Truth 对比

param_names = ['kappa', 'lambda', 'sigma', 'rho', 'v0']
THRESHOLD   = 10.0  # 相对误差超过此阈值时高亮

print("=" * 62)
print("校准结果 vs Ground Truth")
print("=" * 62)
print(f"  {'参数':>8s} | {'真实值':>10s} | {'校准值':>10s} | {'绝对误差':>10s} | {'相对误差':>8s}")
print("  " + "-" * 58)

for n, tv, cv in zip(param_names, true_heston, best_theta):
    ae   = abs(cv - tv)
    re   = ae / (abs(tv) + 1e-8) * 100
    flag = "⚠️ " if re > THRESHOLD else "   "
    print(f"  {flag}{n:>6s} | {tv:10.6f} | {cv:10.6f} | {ae:10.6f} | {re:7.2f}%")

# ── Feller 条件验证 ──
k_cal, l_cal, s_cal = best_theta[0], best_theta[1], best_theta[2]
print()
print(f"  Feller 条件: 2κθ = {2*k_cal*l_cal:.4f} vs σ² = {s_cal**2:.4f}")
print(f"  → {'满足 Feller 条件 ✓' if check_feller(k_cal, l_cal, s_cal) else '不满足 Feller 条件 ✗'}")
print()
print(f"  最终 Vega 加权 IV MSE + Feller = {best_loss:.6e}")

校准结果 vs Ground Truth
        参数 |        真实值 |        校准值 |       绝对误差 |     相对误差
  ----------------------------------------------------------
  ⚠️  kappa |   2.000000 |   3.818759 |   1.818759 |   90.94%
  ⚠️ lambda |   0.100000 |   0.084354 |   0.015646 |   15.65%
  ⚠️  sigma |   0.400000 |   0.546433 |   0.146433 |   36.61%
        rho |  -0.600000 |  -0.549963 |   0.050037 |    8.34%
  ⚠️     v0 |   0.050000 |   0.057086 |   0.007086 |   14.17%

  Feller 条件: 2κθ = 0.6443 vs σ² = 0.2986
  → 满足 Feller 条件 ✓

  最终 Vega 加权 IV MSE + Feller = 3.963323e-05


In [18]:
## Section 13: 用校准后参数重新定价验证（IV 空间 + 价格空间双验证）

M = len(sample_df)

# ── DDN IV 预测（8维输入: 5个Heston参数 + r, tau, log_K_S0） ──
model.eval()
with torch.no_grad():
    cal_theta    = torch.tensor(best_theta, dtype=torch.float32).to(device)
    cal_expanded = cal_theta.unsqueeze(0).expand(M, -1)          # (M, 5)

    r_t    = torch.tensor(sample_df['r'].values,   dtype=torch.float32).to(device)
    tau_t  = torch.tensor(sample_df['tau'].values, dtype=torch.float32).to(device)
    lks_t  = torch.tensor(
        np.log(sample_df['K'].values / sample_df['S0'].values),
        dtype=torch.float32
    ).to(device)

    mkt_feat = torch.stack([r_t, tau_t, lks_t], dim=1)            # (M, 3)
    X_cal    = torch.cat([cal_expanded, mkt_feat], dim=1)          # (M, 8)

    # DDN 现在输出的是 IV
    iv_pred = model(X_cal).cpu().numpy().flatten()                 # 预测 IV

# ── 市场 IV ──
iv_market = sample_df['iv_market'].values

# ── IV 空间误差 ──
iv_abs_err = np.abs(iv_pred - iv_market)
iv_rel_err = iv_abs_err / (np.abs(iv_market) + 1e-8) * 100

print("=" * 65)
print(f"IV 空间验证 ({M} 条)")
print("=" * 65)
print(f"  {'#':>4s} | {'K':>8s} | {'τ':>5s} | {'市场IV':>8s} | {'DDN IV':>8s} | {'IV偏差%':>8s}")
print("  " + "-" * 52)
for i in range(M):
    flag = "✅" if iv_rel_err[i] < 5.0 else "❌"
    print(f"  {flag}{i+1:3d} | {sample_df['K'].iloc[i]:8.1f} | "
          f"{sample_df['tau'].iloc[i]:5.2f} | {iv_market[i]:8.4f} | "
          f"{iv_pred[i]:8.4f} | {iv_rel_err[i]:7.2f}%")

print(f"\n  📊 IV 统计:")
print(f"     平均 IV 相对误差 : {iv_rel_err.mean():.4f}%")
print(f"     最大 IV 相对误差 : {iv_rel_err.max():.4f}%")
print(f"     IV MAE           : {iv_abs_err.mean():.6f}")

# ── 同时做价格空间验证（QuantLib 定价） ──
print(f"\n{'='*65}")
print("价格空间交叉验证（QuantLib 定价）")
print("=" * 65)
P_mkt_arr = sample_df['P_mkt'].values
P_ql  = []
for _, row in sample_df.iterrows():
    p = calculate_heston_price(
        float(best_theta[0]), float(best_theta[1]), float(best_theta[2]),
        float(best_theta[3]), float(best_theta[4]),
        float(row['r']), float(row['tau']), float(row['S0']), float(row['K']),
    )
    P_ql.append(p if not np.isnan(p) else 0.0)
P_ql = np.array(P_ql)

price_rel_err = np.abs(P_ql - P_mkt_arr) / (np.abs(P_mkt_arr) + 1e-8) * 100
print(f"  QL 价格平均误差: {price_rel_err.mean():.4f}%")
print(f"  QL 价格最大误差: {price_rel_err.max():.4f}%")
print()
print("✅ 校准全流程完成！")

IV 空间验证 (99 条)
     # |        K |     τ |     市场IV |   DDN IV |    IV偏差%
  ----------------------------------------------------
  ❌  1 |    600.0 |  0.30 |   0.3159 |   0.3451 |    9.25%
  ❌  2 |    647.4 |  0.30 |   0.3161 |   0.3373 |    6.70%
  ❌  3 |    694.7 |  0.30 |   0.3085 |   0.3283 |    6.43%
  ❌  4 |    742.1 |  0.30 |   0.2990 |   0.3179 |    6.34%
  ❌  5 |    789.5 |  0.30 |   0.2889 |   0.3061 |    5.94%
  ❌  6 |    836.8 |  0.30 |   0.2788 |   0.2932 |    5.17%
  ✅  7 |    884.2 |  0.30 |   0.2688 |   0.2799 |    4.14%
  ✅  8 |    931.6 |  0.30 |   0.2589 |   0.2669 |    3.06%
  ✅  9 |    978.9 |  0.30 |   0.2495 |   0.2549 |    2.17%
  ✅ 10 |   1026.3 |  0.30 |   0.2404 |   0.2445 |    1.71%
  ✅ 11 |   1073.7 |  0.30 |   0.2321 |   0.2364 |    1.87%
  ✅ 12 |   1121.1 |  0.30 |   0.2247 |   0.2307 |    2.66%
  ✅ 13 |   1168.4 |  0.30 |   0.2184 |   0.2268 |    3.86%
  ❌ 14 |   1215.8 |  0.30 |   0.2135 |   0.2244 |    5.13%
  ❌ 15 |   1263.2 |  0.30 |   0.2098 |   0.22

---
## Part 2: 真实 SPY 期权数据 — IV 空间校准 & 跨日定价验证

**流程**:
1. 清洗 `spy_2022_09_01.csv` 和 `spy_2022_09_02.csv` 中的 Call 期权数据
2. **计算市场 IV 和 Vega**（使用 `py_vollib_vectorized`）
3. 校准目标: **Vega 加权 IV MSE + Feller 惩罚**
4. 分别用 9/1 和 9/2 数据校准 Heston 参数
5. 用两组参数分别通过 DDN 和 QuantLib 计算 9/2 的期权价格
6. 在 IV 空间和价格空间双重验证

In [3]:
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d

def interest_rate(date: str, tau: float) -> float: 
    """
    date: '09/01/2022' (确保 CSV 中的日期格式一致)
    tau: 剩余期限（年化，如 45/365）
    """
    # 期限节点 (年化): 1mo, 2mo, 3mo, 6mo, 1yr
    maturities = np.array([1/12, 2/12, 3/12, 6/12, 1.0]) 

    # 1. 读取数据 (假设 CSV 文件在当前目录下)
    data = pd.read_csv(ROOT / "data" / "par-yield-curve-rates-2020-2023.csv")
    
    # 2. 筛选特定日期并提取数值
    ir_row = data[data['date'] == date]
    if ir_row.empty:
        raise ValueError(f"未找到日期为 {date} 的数据，请检查日期格式是否为 YYYY-MM-DD 或 MM/DD/YYYY")

    # 关键修正：使用 .values.flatten() 将 DataFrame 的一行转为 1D 数组
    # 并且除以 100 将百分比转为小数 (例如 5.2 -> 0.052)
    market_rates = ir_row[['1 mo', '2 mo', '3 mo', '6 mo', '1 yr']].values.flatten() / 100.0

    # 步骤 1: 将市场单利转换为连续复利
    # 公式: r_c = ln(1 + R * t) / t 
    continuous_rates = np.log(1 + market_rates * maturities) / maturities

    # 步骤 2: 构建插值函数 (推荐在点数较少时先试用 'linear'，'cubic' 曲线更平滑但点少时易震荡)
    yield_curve = interp1d(maturities, continuous_rates, kind='cubic', fill_value="extrapolate")

    return float(yield_curve(tau))

# 3. 关键修正：if __name__ == '__main__': (去掉引号)
if __name__ == '__main__':
    # 假设期权链中有一个期权还有 45 天到期
    tau = 0.25
    try:
        exact_r = interest_rate('09/01/2022', tau)
        print(f"45天到期期权适用的连续复利无风险利率为: {exact_r:.6f}")
        # 输出示例: 0.024869 (即 2.4869%)
    except Exception as e:
        print(f"运行错误: {e}")

45天到期期权适用的连续复利无风险利率为: 0.029590


In [16]:
## Section 14: 清洗 SPY 真实期权 CSV 数据 + 计算市场 IV & Vega

import pandas as pd
import numpy as np
from datetime import datetime
from pathlib import Path
from modules.pricing import compute_iv_vega_batch

DATA_DIR = Path.cwd() / "data"

# ── 筛选参数 ──
DTE_LO, DTE_HI       = 40, 300          # 到期天数范围
LOG_KS_LO, LOG_KS_HI = -0.15, 0.2  # log-moneyness 范围

def load_and_clean_spy(csv_path: str | Path, label: str = "") -> pd.DataFrame:
    """
    清洗 SPY 期权 CSV → 计算市场 IV & Vega。

    输出列: r, tau, S0, K, P_mkt, log_K_S0, iv_market, vega
    仅保留 Call 期权且 IV/Vega 有效的行。
    """
    dt_obj = datetime.strptime(label, "%Y-%m-%d")
    date = dt_obj.strftime("%m/%d/%Y")

    maturities = np.array([1/12, 2/12, 3/12, 6/12, 1.0])
    yield_data = pd.read_csv(ROOT / "data" / "par-yield-curve-rates-2020-2023.csv")
    ir_row = yield_data[yield_data['date'] == date]
    if ir_row.empty:
        raise ValueError(f"未找到日期为 {date} 的数据")
    market_rates = ir_row[['1 mo', '2 mo', '3 mo', '6 mo', '1 yr']].values.flatten() / 100.0
    continuous_rates = np.log(1 + market_rates * maturities) / maturities
    yield_curve = interp1d(maturities, continuous_rates, kind='cubic', fill_value="extrapolate")

    # 读取和清洗 CSV
    raw = pd.read_csv(csv_path)
    raw.columns = [c.strip().strip("[]") for c in raw.columns]
    for col in ["UNDERLYING_LAST", "DTE", "STRIKE", "C_LAST", "C_BID", "C_ASK"]:
        raw[col] = pd.to_numeric(raw[col], errors="coerce")
    # C_VOLUME 在 CSV 中为 str 类型且存在空字符串/null，需先转数值再筛选
    raw["C_VOLUME"] = pd.to_numeric(raw["C_VOLUME"], errors="coerce")

    df = raw.dropna(subset=["UNDERLYING_LAST", "DTE", "STRIKE", "C_LAST"]).copy()
    df = df[df["C_LAST"] > 0.01].copy()
    df = df[df["UNDERLYING_LAST"] > 0].copy()
    df = df[df["C_VOLUME"].fillna(0) > 0].copy()   # 确保有交易量（NaN 视为 0）
    df = df[(df["DTE"] >= DTE_LO) & (df["DTE"] <= DTE_HI)].copy()

    df["S0"]       = df["UNDERLYING_LAST"]
    df["K"]        = df["STRIKE"]
    df["P_mkt"]    = (df["C_BID"] + df["C_ASK"]) * 0.5
    df["tau"]      = df["DTE"] / 365.0
    df["r"]        = df["tau"].apply(lambda t: interest_rate(date, t))
    df["log_K_S0"] = np.log(df["K"] / df["S0"])

    df = df[(df["log_K_S0"] >= LOG_KS_LO) & (df["log_K_S0"] <= LOG_KS_HI)].copy()

    # ── 计算市场 IV 和 Vega（核心升级） ──
    iv_arr, vega_arr = compute_iv_vega_batch(
        df["P_mkt"].values,
        df["S0"].values,
        df["K"].values,
        df["tau"].values,
        df["r"].values,
    )
    df["iv_market"] = iv_arr
    df["vega"]      = vega_arr

    # 过滤 IV/Vega 为 NaN 或 ≤ 0 的脏数据（违反无套利边界）
    n_before = len(df)
    df = df.dropna(subset=["iv_market", "vega"])
    df = df[(df["iv_market"] > 0) & (df["vega"] > 0)].copy()
    n_dropped = n_before - len(df)

    out = df[["r", "tau", "S0", "K", "P_mkt", "log_K_S0", "iv_market", "vega"]].reset_index(drop=True)

    # 从中间截取最多 100 条用于校准（自适应，防止越界）
    n_take = min(100, len(out))
    mid    = len(out) // 2
    start  = max(0, mid - n_take // 2)
    out    = out.iloc[start : start + n_take].reset_index(drop=True)

    print(f"[{label}] 清洗完成: {len(out)} 条有效 Call 期权 (丢弃 {n_dropped} 条 IV 无效)")
    if len(out) == 0:
        raise ValueError(f"[{label}] 清洗后无有效数据，请检查筛选条件")
    print(f"  S0 = {out['S0'].iloc[0]:.2f},  DTE ∈ [{out['tau'].min()*365:.0f}, {out['tau'].max()*365:.0f}] 天")
    print(f"  r ∈ [{out['r'].min():.4f}, {out['r'].max():.4f}]")
    print(f"  K  ∈ [{out['K'].min():.1f}, {out['K'].max():.1f}]")
    print(f"  IV ∈ [{out['iv_market'].min():.4f}, {out['iv_market'].max():.4f}]")
    print(f"  Vega ∈ [{out['vega'].min():.2f}, {out['vega'].max():.2f}]")
    return out

# ── 加载两天的数据 ──
df_sep1 = load_and_clean_spy(DATA_DIR / "spy_2022_09_01.csv", label="2022-09-01")
print("示例:")
display(df_sep1.head(5))
df_sep2 = load_and_clean_spy(DATA_DIR / "spy_2022_09_02.csv", label="2022-09-02")
print("示例:")
display(df_sep2.head(5))

[2022-09-01] 清洗完成: 100 条有效 Call 期权 (丢弃 0 条 IV 无效)
  S0 = 396.39,  DTE ∈ [78, 106] 天
  r ∈ [0.0289, 0.0303]
  K  ∈ [350.0, 480.0]
  IV ∈ [0.1717, 0.2621]
  Vega ∈ [0.05, 0.85]
示例:


,r,tau,S0,K,P_mkt,log_K_S0,iv_market,vega
0,0.028941,0.213808,396.39,412.0,9.32,0.038625,0.201290,0.698525
1,0.028941,0.213808,396.39,414.0,8.50,0.043467,0.199098,0.685468
2,0.028941,0.213808,396.39,415.0,8.10,0.045880,0.197909,0.678046
3,0.028941,0.213808,396.39,416.0,7.72,0.048287,0.196854,0.670154
4,0.028941,0.213808,396.39,418.0,6.99,0.053083,0.194704,0.652724


[2022-09-02] 清洗完成: 100 条有效 Call 期权 (丢弃 1 条 IV 无效)
  S0 = 392.27,  DTE ∈ [77, 105] 天
  r ∈ [0.0287, 0.0299]
  K  ∈ [340.0, 475.0]
  IV ∈ [0.1741, 0.2501]
  Vega ∈ [0.05, 0.84]
示例:


/Users/liaojiansong/anaconda3/lib/python3.13/site-packages/py_vollib_vectorized/implied_volatility.py:75: UserWarning: Found Below Intrinsic contracts at index [51]
  below_intrinsic, above_max_price = _check_below_and_above_intrinsic(K, F, flag, undiscounted_option_price, on_error)


,r,tau,S0,K,P_mkt,log_K_S0,iv_market,vega
0,0.028698,0.211068,392.27,405.0,10.415,0.031937,0.206015,0.700757
1,0.028698,0.211068,392.27,406.0,9.970,0.034403,0.204896,0.696058
2,0.028698,0.211068,392.27,408.0,9.120,0.039317,0.202765,0.685034
3,0.028698,0.211068,392.27,410.0,8.325,0.044207,0.200807,0.671907
4,0.028698,0.211068,392.27,412.0,7.555,0.049073,0.198574,0.656454


In [13]:
## Section 15: 分别用 9/1 和 9/2 数据校准 Heston 参数（IV + Feller）

from modules.calibration import HestonCalibrator
from modules.pricing import check_feller
import time

calibrator = HestonCalibrator(model, device, seed=42)

def build_market_dict(df: pd.DataFrame) -> dict:
    """构建 market_data 字典（包含 IV 和 Vega）。"""
    return {
        "r":         df["r"].values.astype(np.float32),
        "tau":       df["tau"].values.astype(np.float32),
        "S0":        df["S0"].values.astype(np.float32),
        "K":         df["K"].values.astype(np.float32),
        "iv_market": df["iv_market"].values,    # ★ 市场 IV
        "vega":      df["vega"].values,          # ★ Vega 权重
    }

# ── 9月1日 校准 ──
t0 = time.time()
print("=" * 60)
print("校准 — 2022-09-01 SPY 期权数据 (IV 空间 + Feller)")
print("=" * 60)
theta_sep1, loss_sep1, _ = calibrator.calibrate(
    build_market_dict(df_sep1),
    n_starts=10, lr=5e-3, max_steps=300, patience=50,
    lambda_feller=10.0, verbose=True,
)
T0 = time.time() - t0

# ── 9月2日 校准 ──
t1 = time.time()
print("\n" + "=" * 60)
print("校准 — 2022-09-02 SPY 期权数据 (IV 空间 + Feller)")
print("=" * 60)
theta_sep2, loss_sep2, _ = calibrator.calibrate(
    build_market_dict(df_sep2),
    n_starts=10, lr=5e-3, max_steps=300, patience=50,
    lambda_feller=10.0, verbose=True,
)
T1 = time.time() - t1

# ── 对比两天校准结果 ──
param_names = ["kappa", "lambda", "sigma", "rho", "v0"]
print("\n" + "=" * 60)
print("两天校准参数对比")
print("=" * 60)
print(f"  {'参数':>8s} | {'9/1 校准值':>12s} | {'9/2 校准值':>12s} | {'变化量':>10s}")
print("  " + "-" * 52)
for n, v1, v2 in zip(param_names, theta_sep1, theta_sep2):
    diff = v2 - v1
    print(f"  {n:>8s} | {v1:12.6f} | {v2:12.6f} | {diff:+10.6f}")

# ── Feller 条件检查 ──
for label, theta in [("9/1", theta_sep1), ("9/2", theta_sep2)]:
    k, l, s = theta[0], theta[1], theta[2]
    ok = check_feller(k, l, s)
    print(f"\n  [{label}] Feller: 2κθ={2*k*l:.4f} vs σ²={s**2:.4f} → {'满足 ✓' if ok else '不满足 ✗'}")

print(f"\n  9/1 loss = {loss_sep1:.6e}  (耗时 {T0:.1f}s)")
print(f"  9/2 loss = {loss_sep2:.6e}  (耗时 {T1:.1f}s)")

校准 — 2022-09-01 SPY 期权数据 (IV 空间 + Feller)
  市场期权数   : 100
  初始点数     : 10
  Adam lr      : 0.005，max_steps=300，patience=50
  Feller λ     : 10.0
  market IV    : [0.1717, 0.2621]

  [start  1/10] step= 100 | loss=8.4601e-05 | Feller=✓(0.00e+00) | lr=5.00e-03
  [start  1/10] step= 200 | loss=5.3904e-05 | Feller=✓(0.00e+00) | lr=5.00e-03
  [start  1/10] step= 300 | loss=4.2010e-05 | Feller=✓(0.00e+00) | lr=5.00e-03
  [start  1/10] loss=4.2010e-05  [κ=3.6123, λ=0.1605, σ=1.0000, ρ=-0.6063, v₀=0.0186] Feller✓ ★
  [start  2/10] step= 100 | loss=1.2791e-02 | Feller=✓(0.00e+00) | lr=5.00e-03
  [start  2/10] step= 200 | loss=1.7881e-04 | Feller=✓(0.00e+00) | lr=5.00e-03
  [start  2/10] step= 300 | loss=1.3224e-04 | Feller=✓(0.00e+00) | lr=5.00e-03
  [start  2/10] loss=1.3224e-04  [κ=4.2889, λ=0.1662, σ=0.9712, ρ=-0.8595, v₀=0.0102] Feller✓
  [start  3/10] step= 100 | loss=2.1613e-03 | Feller=✓(0.00e+00) | lr=5.00e-03
  [start  3/10] step= 200 | loss=1.4039e-04 | Feller=✓(0.00e+00) | lr=5.00e-0

In [14]:
## Section 16: 用校准参数对 9/2 数据定价 — 跨日验证 vs 当日验证 汇总

from modules.pricing import calculate_heston_price, compute_iv_vega_batch

M       = len(df_sep2)
P_real  = df_sep2["P_mkt"].values       # 9/2 真实价格
iv_real = df_sep2["iv_market"].values   # 9/2 真实 IV

# ── 辅助: DDN IV 批量预测（8维输入） ────────────────────────────────
def ddn_iv_batch(theta_np: np.ndarray, df: pd.DataFrame) -> np.ndarray:
    n = len(df)
    model.eval()
    with torch.no_grad():
        th   = torch.tensor(theta_np, dtype=torch.float32, device=device)
        th_e = th.unsqueeze(0).expand(n, -1)
        r_t   = torch.tensor(df["r"].values,   dtype=torch.float32, device=device)
        tau_t = torch.tensor(df["tau"].values,  dtype=torch.float32, device=device)
        lks_t = torch.tensor(df["log_K_S0"].values, dtype=torch.float32, device=device)
        mkt   = torch.stack([r_t, tau_t, lks_t], dim=1)
        X     = torch.cat([th_e, mkt], dim=1)
        return model(X).cpu().numpy().flatten()

# ── 辅助: QuantLib 批量定价 ─────────────────────────────────────────
def ql_price_batch(theta_np: np.ndarray, df: pd.DataFrame) -> np.ndarray:
    kappa, lam, sigma, rho, v0 = theta_np
    prices = []
    for _, row in df.iterrows():
        p = calculate_heston_price(
            float(kappa), float(lam), float(sigma), float(rho), float(v0),
            float(row["r"]), float(row["tau"]), float(row["S0"]), float(row["K"]),
        )
        prices.append(p if not np.isnan(p) else 0.0)
    return np.array(prices)

def ql_iv_batch(theta_np: np.ndarray, df: pd.DataFrame) -> np.ndarray:
    prices = ql_price_batch(theta_np, df)
    iv_arr, _ = compute_iv_vega_batch(
        prices, df["S0"].values, df["K"].values,
        df["tau"].values, df["r"].values,
    )
    return iv_arr

def calc_mre(pred, true):
    return float(np.nanmean(np.abs(pred - true) / (np.abs(true) + 1e-8)))

# ── 计算四组预测 ─────────────────────────────────────────────────────
print("正在计算 DDN IV 预测...")
iv_ddn_sep1 = ddn_iv_batch(theta_sep1, df_sep2)   # 9/1参数 → 9/2（跨日）
iv_ddn_sep2 = ddn_iv_batch(theta_sep2, df_sep2)   # 9/2参数 → 9/2（当日）

print("正在计算 QuantLib 定价 + IV...")
P_ql_sep1  = ql_price_batch(theta_sep1, df_sep2)
P_ql_sep2  = ql_price_batch(theta_sep2, df_sep2)
iv_ql_sep1 = ql_iv_batch(theta_sep1, df_sep2)
iv_ql_sep2 = ql_iv_batch(theta_sep2, df_sep2)

# ── 汇总 MRE 表 ──────────────────────────────────────────────────────
print("\n" + "=" * 68)
print(f"汇总误差对比 — 9/2 真实期权 ({M} 条)")
print("=" * 68)
print(f"  {'方法':>22s} | {'校准数据':>8s} | {'IV MRE':>10s} | {'Price MRE':>11s}")
print("  " + "-" * 60)

rows_summary = [
    ("DDN",          "9/1→9/2 跨日", iv_ddn_sep1, None),
    ("DDN",          "9/2→9/2 当日", iv_ddn_sep2, None),
    ("QuantLib→IV",  "9/1→9/2 跨日", iv_ql_sep1,  P_ql_sep1),
    ("QuantLib→IV",  "9/2→9/2 当日", iv_ql_sep2,  P_ql_sep2),
]
for method, scenario, iv_pred, p_pred in rows_summary:
    iv_mre = calc_mre(iv_pred, iv_real)
    p_mre  = calc_mre(p_pred, P_real) if p_pred is not None else float("nan")
    p_str  = f"{p_mre:11.4f}" if not np.isnan(p_mre) else f"{'—':>11s}"
    print(f"  {method:>22s} | {scenario:>8s} | {iv_mre:10.4f} | {p_str}")

print(f"\n  说明: IV MRE = mean|IV_pred − IV_mkt| / |IV_mkt|")

正在计算 DDN IV 预测...
正在计算 QuantLib 定价 + IV...

汇总误差对比 — 9/2 真实期权 (100 条)
                      方法 |     校准数据 |     IV MRE |   Price MRE
  ------------------------------------------------------------
                     DDN | 9/1→9/2 跨日 |     0.0198 |           —
                     DDN | 9/2→9/2 当日 |     0.0156 |           —
             QuantLib→IV | 9/1→9/2 跨日 |     0.0923 |      0.1889
             QuantLib→IV | 9/2→9/2 当日 |     0.0815 |      0.1810

  说明: IV MRE = mean|IV_pred − IV_mkt| / |IV_mkt|


In [15]:
## Section 17: 逐条 IV 明细 & 误差分布（跨日验证 vs 当日验证）

def print_iv_detail(title, df_tgt, iv_pred_ddn, iv_pred_ql, p_pred_ql,
                    iv_true, p_true, n_show=50):
    """打印逐条明细 + 误差分布统计。"""
    M = len(iv_true)
    err_ddn = np.abs(iv_pred_ddn - iv_true) / (np.abs(iv_true) + 1e-8) * 100
    err_ql  = np.abs(iv_pred_ql  - iv_true) / (np.abs(iv_true) + 1e-8) * 100

    print("=" * 92)
    print(f"  {title}  (共 {M} 条，显示前 {min(n_show, M)} 条)")
    print("=" * 92)
    print(f"  {'#':>4s} | {'K':>8s} | {'τ':>6s} | {'IV_mkt':>8s} | "
          f"{'DDN_IV':>8s} | {'QL_IV':>8s} | {'DDN%':>7s} | {'QL%':>7s} | "
          f"{'P_mkt':>8s} | {'P_QL':>8s}")
    print("  " + "-" * 88)
    for i in range(min(n_show, M)):
        flag = "✅" if err_ddn[i] < 5.0 else "❌"
        print(f"  {flag}{i+1:3d} | {df_tgt['K'].iloc[i]:8.1f} | "
              f"{df_tgt['tau'].iloc[i]:6.3f} | {iv_true[i]:8.4f} | "
              f"{iv_pred_ddn[i]:8.4f} | {iv_pred_ql[i]:8.4f} | "
              f"{err_ddn[i]:6.2f}% | {err_ql[i]:6.2f}% | "
              f"{p_true[i]:8.2f} | {p_pred_ql[i]:8.2f}")

    print(f"\n  {'─'*60}")
    print(f"  误差分布统计:")
    print(f"  {'─'*60}")
    print(f"  {'指标':>20s} | {'DDN IV':>10s} | {'QL IV':>10s}")
    print(f"  {'─'*48}")
    stats = [
        ("均值 MRE (%)",   np.nanmean(err_ddn),   np.nanmean(err_ql)),
        ("中位数 (%)",     np.nanmedian(err_ddn),  np.nanmedian(err_ql)),
        ("最大值 (%)",     np.nanmax(err_ddn),     np.nanmax(err_ql)),
        ("< 1% 占比 (%)",  (err_ddn < 1.0).mean()*100,  (err_ql < 1.0).mean()*100),
        ("< 5% 占比 (%)",  (err_ddn < 5.0).mean()*100,  (err_ql < 5.0).mean()*100),
        ("< 10% 占比 (%)", (err_ddn < 10.0).mean()*100, (err_ql < 10.0).mean()*100),
    ]
    for name, v_ddn, v_ql in stats:
        print(f"  {name:>20s} | {v_ddn:10.4f} | {v_ql:10.4f}")
    print()

# ── 跨日验证: 9/1 校准参数 → 9/2 期权 ──────────────────────────────
print_iv_detail(
    title        = "【跨日验证】9/1 校准参数 → 预测 9/2 IV",
    df_tgt       = df_sep2,
    iv_pred_ddn  = iv_ddn_sep1,
    iv_pred_ql   = iv_ql_sep1,
    p_pred_ql    = P_ql_sep1,
    iv_true      = iv_real,
    p_true       = P_real,
)

# ── 当日验证: 9/2 校准参数 → 9/2 期权 ──────────────────────────────
print_iv_detail(
    title        = "【当日验证】9/2 校准参数 → 预测 9/2 IV",
    df_tgt       = df_sep2,
    iv_pred_ddn  = iv_ddn_sep2,
    iv_pred_ql   = iv_ql_sep2,
    p_pred_ql    = P_ql_sep2,
    iv_true      = iv_real,
    p_true       = P_real,
)

print("✅ 真实 SPY 数据 IV 空间校准 & 跨日/当日验证完成！")

  【跨日验证】9/1 校准参数 → 预测 9/2 IV  (共 100 条，显示前 50 条)
     # |        K |      τ |   IV_mkt |   DDN_IV |    QL_IV |    DDN% |     QL% |    P_mkt |     P_QL
  ----------------------------------------------------------------------------------------
  ✅  1 |    405.0 |  0.211 |   0.2060 |   0.2027 |   0.2242 |   1.59% |   8.81% |    10.41 |    11.69
  ✅  2 |    406.0 |  0.211 |   0.2049 |   0.2017 |   0.2232 |   1.57% |   8.91% |     9.97 |    11.25
  ✅  3 |    408.0 |  0.211 |   0.2028 |   0.1996 |   0.2211 |   1.56% |   9.06% |     9.12 |    10.38
  ✅  4 |    410.0 |  0.211 |   0.2008 |   0.1976 |   0.2191 |   1.61% |   9.10% |     8.32 |     9.56
  ✅  5 |    412.0 |  0.211 |   0.1986 |   0.1956 |   0.2170 |   1.51% |   9.30% |     7.55 |     8.78
  ✅  6 |    414.0 |  0.211 |   0.1965 |   0.1936 |   0.2150 |   1.47% |   9.39% |     6.84 |     8.03
  ✅  7 |    415.0 |  0.211 |   0.1955 |   0.1927 |   0.2139 |   1.46% |   9.42% |     6.50 |     7.68
  ✅  8 |    416.0 |  0.211 |   0.1945 |   0.


---
## Part 3: NVDA 真实期权数据 — 2021-11-01 / 2021-11-02 当日与跨日校准

该新增部分只使用 NVDA 数据，不依赖前面 SPY cell 的运行状态；会重新加载已保存模型权重、清洗两天的 NVDA call option、匹配利率曲线，并同时运行 `Heston_Project` 与 `Zhang_realize` 两个模型，最后在 `experiments result/` 下输出 CSV/JSON 和 LaTeX 表格。


In [ ]:

# NVDA 2021-11-01 / 2021-11-02 calibration block
# Self-contained: do not rerun earlier SPY cells.

from __future__ import annotations

import os
os.environ.setdefault("NUMBA_DISABLE_JIT", "1")

import sys
import json
import math
import time
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
from scipy.interpolate import interp1d
import py_vollib_vectorized as pvv

PROJECT_DIR = Path.cwd().resolve()
ROOT = PROJECT_DIR.parent if PROJECT_DIR.name in {"Heston_Project", "Zhang_realize"} else PROJECT_DIR
NVDA_CSV = ROOT / "nvda_2020_2022.csv"
OUT_DIR = ROOT / "experiments result"
OUT_DIR.mkdir(exist_ok=True)

DATES = ("2021-11-01", "2021-11-02")
N_CONTRACTS = 10
DTE_LO, DTE_HI = 40, 300
LOG_KS_LO, LOG_KS_HI = -0.15, 0.20


def pct_tex(x: float | None) -> str:
    if x is None or (isinstance(x, float) and (math.isnan(x) or math.isinf(x))):
        return "n/a"
    return f"{100.0 * float(x):.2f}\\%"


def pct_text(x: float | None) -> str:
    if x is None or (isinstance(x, float) and (math.isnan(x) or math.isinf(x))):
        return "n/a"
    return f"{100.0 * float(x):.2f}%"


def matched_interest_rate(yield_data: pd.DataFrame, quote_date: str, tau: float) -> float:
    date = pd.to_datetime(quote_date).strftime("%m/%d/%Y")
    maturities = np.array([1 / 12, 2 / 12, 3 / 12, 6 / 12, 1.0], dtype=float)
    row = yield_data[yield_data["date"] == date]
    if row.empty:
        raise ValueError(f"No yield curve row for {date}")
    par_rates = row[["1 mo", "2 mo", "3 mo", "6 mo", "1 yr"]].values.flatten().astype(float) / 100.0
    continuous_rates = np.log(1.0 + par_rates * maturities) / maturities
    curve = interp1d(maturities, continuous_rates, kind="cubic", fill_value="extrapolate")
    return float(curve(tau))


def compute_iv_vega(price, s0, strike, tau, rate):
    price_s = pd.Series(np.asarray(price, dtype=float))
    s0_s = pd.Series(np.asarray(s0, dtype=float))
    strike_s = pd.Series(np.asarray(strike, dtype=float))
    tau_s = pd.Series(np.asarray(tau, dtype=float))
    rate_s = pd.Series(np.asarray(rate, dtype=float))
    flag_s = pd.Series(["c"] * len(price_s))
    iv_s = pvv.vectorized_implied_volatility(
        price_s,
        s0_s,
        strike_s,
        tau_s,
        rate_s,
        flag_s,
        q=0.0,
        model="black_scholes_merton",
        return_as="series",
    )
    vega_s = pvv.vectorized_vega(
        flag_s,
        s0_s,
        strike_s,
        tau_s,
        rate_s,
        iv_s,
        q=0.0,
        model="black_scholes_merton",
        return_as="series",
    )
    return iv_s.values.astype(float), vega_s.values.astype(float)


def load_nvda_rows() -> pd.DataFrame:
    needed = {"QUOTE_DATE", "UNDERLYING_LAST", "DTE", "STRIKE", "C_LAST", "C_BID", "C_ASK", "C_VOLUME"}
    pieces = []
    for chunk in pd.read_csv(
        NVDA_CSV,
        chunksize=200_000,
        dtype=str,
        usecols=lambda c: c.strip().strip("[]") in needed,
    ):
        chunk.columns = [c.strip().strip("[]") for c in chunk.columns]
        chunk["QUOTE_DATE"] = chunk["QUOTE_DATE"].astype(str).str.strip()
        mask = chunk["QUOTE_DATE"].isin(DATES)
        if mask.any():
            pieces.append(chunk.loc[mask].copy())
    if not pieces:
        raise ValueError("No NVDA rows found for 2021-11-01 and 2021-11-02")
    return pd.concat(pieces, ignore_index=True)


_RAW_NVDA = load_nvda_rows()


def load_and_clean_nvda(yield_data: pd.DataFrame, label: str) -> pd.DataFrame:
    df = _RAW_NVDA[_RAW_NVDA["QUOTE_DATE"] == label].copy()
    for col in ["UNDERLYING_LAST", "DTE", "STRIKE", "C_LAST", "C_BID", "C_ASK", "C_VOLUME"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.dropna(subset=["UNDERLYING_LAST", "DTE", "STRIKE", "C_LAST", "C_BID", "C_ASK"]).copy()
    df = df[df["C_LAST"] > 0.01].copy()
    df = df[df["UNDERLYING_LAST"] > 0].copy()
    df = df[df["C_VOLUME"].fillna(0) > 0].copy()
    df = df[(df["DTE"] >= DTE_LO) & (df["DTE"] <= DTE_HI)].copy()

    df["S0"] = df["UNDERLYING_LAST"]
    df["K"] = df["STRIKE"]
    df["P_mkt"] = (df["C_BID"] + df["C_ASK"]) * 0.5
    df["tau"] = df["DTE"] / 365.0
    df["log_K_S0"] = np.log(df["K"] / df["S0"])
    df = df[(df["log_K_S0"] >= LOG_KS_LO) & (df["log_K_S0"] <= LOG_KS_HI)].copy()
    df = df[df["P_mkt"] > 0].copy()

    df["r"] = df["tau"].map(lambda t: matched_interest_rate(yield_data, label, float(t)))
    iv_arr, vega_arr = compute_iv_vega(df["P_mkt"].values, df["S0"].values, df["K"].values, df["tau"].values, df["r"].values)
    df["iv_market"] = iv_arr
    df["vega"] = vega_arr
    df = df.dropna(subset=["iv_market", "vega"]).copy()
    df = df[(df["iv_market"] > 0) & (df["vega"] > 0)].copy()

    df["rel_spread"] = (df["C_ASK"] - df["C_BID"]) / df["P_mkt"].replace(0, np.nan)
    df["abs_log_K_S0"] = df["log_K_S0"].abs()
    df = df.sort_values(
        ["abs_log_K_S0", "rel_spread", "C_VOLUME", "DTE", "K"],
        ascending=[True, True, False, True, True],
    ).head(N_CONTRACTS)
    df = df.sort_values(["DTE", "K"]).reset_index(drop=True)

    cols = ["r", "tau", "S0", "K", "P_mkt", "log_K_S0", "iv_market", "vega", "DTE", "C_VOLUME", "rel_spread"]
    out = df[cols].copy()
    if len(out) < N_CONTRACTS:
        raise ValueError(f"{label}: only {len(out)} valid contracts after cleaning")
    print(
        f"[{label}] selected {len(out)} NVDA calls: "
        f"S0={out['S0'].iloc[0]:.2f}, "
        f"K=[{out['K'].min():.2f}, {out['K'].max():.2f}], "
        f"DTE=[{out['DTE'].min():.0f}, {out['DTE'].max():.0f}], "
        f"IV=[{out['iv_market'].min():.4f}, {out['iv_market'].max():.4f}]"
    )
    return out


def clear_project_modules():
    for name in list(sys.modules):
        if name == "modules" or name.startswith("modules."):
            del sys.modules[name]


def load_project_model(project_root: Path, default_input_dim: int) -> dict:
    clear_project_modules()
    sys.path.insert(0, str(project_root))
    cwd = Path.cwd()
    os.chdir(project_root)
    try:
        from modules.model import HestonDDN
        from modules.calibration import HestonCalibrator
        from modules.pricing import calculate_heston_price
        try:
            from modules.pricing import check_feller
        except Exception:
            check_feller = lambda kappa, theta, sigma: 2.0 * kappa * theta > sigma**2

        model_path = project_root / "models" / "heston_ddn_weights.pth"
        device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")
        ckpt = torch.load(model_path, map_location=device)
        model = HestonDDN(input_dim=ckpt.get("input_dim", default_input_dim), heston_param_dim=ckpt.get("heston_dim", 5)).to(device)
        model.load_state_dict(ckpt["model_state_dict"])
        model.is_fitted = ckpt.get("is_fitted", True)
        model.eval()
        return {
            "device": device,
            "model": model,
            "calibrator_cls": HestonCalibrator,
            "calculate_heston_price": calculate_heston_price,
            "check_feller": check_feller,
        }
    finally:
        os.chdir(cwd)
        try:
            sys.path.remove(str(project_root))
        except ValueError:
            pass


def market_dict_heston(df: pd.DataFrame) -> dict:
    return {
        "r": df["r"].values.astype(np.float32),
        "tau": df["tau"].values.astype(np.float32),
        "S0": df["S0"].values.astype(np.float32),
        "K": df["K"].values.astype(np.float32),
        "iv_market": df["iv_market"].values.astype(np.float64),
        "vega": df["vega"].values.astype(np.float64),
    }


def market_dict_zhang(df: pd.DataFrame) -> dict:
    return {
        "r": df["r"].values.astype(np.float32),
        "tau": df["tau"].values.astype(np.float32),
        "S0": df["S0"].values.astype(np.float32),
        "K": df["K"].values.astype(np.float32),
        "P_mkt": df["P_mkt"].values.astype(np.float32),
    }


def calc_mre(pred, true) -> float:
    pred = np.asarray(pred, dtype=float)
    true = np.asarray(true, dtype=float)
    valid = np.isfinite(pred) & np.isfinite(true) & (np.abs(true) > 1e-12)
    if not valid.any():
        return float("nan")
    return float(np.mean(np.abs(pred[valid] - true[valid]) / np.abs(true[valid])))


def error_stats(pred, true) -> dict:
    pred = np.asarray(pred, dtype=float)
    true = np.asarray(true, dtype=float)
    valid = np.isfinite(pred) & np.isfinite(true) & (np.abs(true) > 1e-12)
    err = np.full_like(true, np.nan, dtype=float)
    err[valid] = np.abs(pred[valid] - true[valid]) / np.abs(true[valid])
    return {
        "mre": float(np.nanmean(err)),
        "median": float(np.nanmedian(err)),
        "max": float(np.nanmax(err)),
        "lt_1pct": float(np.nanmean(err < 0.01)),
        "lt_5pct": float(np.nanmean(err < 0.05)),
        "lt_10pct": float(np.nanmean(err < 0.10)),
        "valid": int(valid.sum()),
        "total": int(len(true)),
    }


def ql_price_batch(calc_price, theta, df: pd.DataFrame) -> np.ndarray:
    kappa, theta_bar, sigma, rho, v0 = [float(x) for x in theta]
    prices = []
    for _, row in df.iterrows():
        p = calc_price(kappa, theta_bar, sigma, rho, v0, float(row["r"]), float(row["tau"]), float(row["S0"]), float(row["K"]))
        prices.append(p if np.isfinite(p) else np.nan)
    return np.asarray(prices, dtype=float)


def iv_from_prices(prices, df: pd.DataFrame) -> np.ndarray:
    iv_arr, _ = compute_iv_vega(prices, df["S0"].values, df["K"].values, df["tau"].values, df["r"].values)
    return iv_arr


def ddn_iv_batch(model, device, theta, df: pd.DataFrame) -> np.ndarray:
    n = len(df)
    model.eval()
    with torch.no_grad():
        theta_t = torch.tensor(theta, dtype=torch.float32, device=device).unsqueeze(0).expand(n, -1)
        r_t = torch.tensor(df["r"].values, dtype=torch.float32, device=device)
        tau_t = torch.tensor(df["tau"].values, dtype=torch.float32, device=device)
        lks_t = torch.tensor(df["log_K_S0"].values, dtype=torch.float32, device=device)
        x = torch.cat([theta_t, torch.stack([r_t, tau_t, lks_t], dim=1)], dim=1)
        return model(x).cpu().numpy().flatten()


def ddn_price_batch(model, device, theta, df: pd.DataFrame) -> np.ndarray:
    n = len(df)
    model.eval()
    with torch.no_grad():
        theta_t = torch.tensor(theta, dtype=torch.float32, device=device).unsqueeze(0).expand(n, -1)
        s0_t = torch.tensor(df["S0"].values, dtype=torch.float32, device=device)
        r_t = torch.tensor(df["r"].values, dtype=torch.float32, device=device)
        tau_t = torch.tensor(df["tau"].values, dtype=torch.float32, device=device)
        lks_t = torch.tensor(df["log_K_S0"].values, dtype=torch.float32, device=device)
        x = torch.cat([theta_t, torch.stack([r_t, tau_t, s0_t, lks_t], dim=1)], dim=1)
        p_norm = model(x).cpu().numpy().flatten()
    return p_norm * df["S0"].values.astype(float)


def run_heston_project(df_1101: pd.DataFrame, df_1102: pd.DataFrame) -> dict:
    project = load_project_model(ROOT / "Heston_Project", default_input_dim=8)
    model, device = project["model"], project["device"]
    calibrator = project["calibrator_cls"](model, device, seed=42)

    t0 = time.time()
    theta_1101, loss_1101, _ = calibrator.calibrate(
        market_dict_heston(df_1101),
        n_starts=10,
        lr=5e-3,
        max_steps=300,
        patience=50,
        lambda_feller=10.0,
        verbose=False,
    )
    time_1101 = time.time() - t0

    t0 = time.time()
    theta_1102, loss_1102, _ = calibrator.calibrate(
        market_dict_heston(df_1102),
        n_starts=10,
        lr=5e-3,
        max_steps=300,
        patience=50,
        lambda_feller=10.0,
        verbose=False,
    )
    time_1102 = time.time() - t0

    iv_true = df_1102["iv_market"].values
    price_true = df_1102["P_mkt"].values
    calc_price = project["calculate_heston_price"]
    scenarios = {}
    for scenario, theta in [("cross_day", theta_1101), ("same_day", theta_1102)]:
        ddn_iv = ddn_iv_batch(model, device, theta, df_1102)
        ql_price = ql_price_batch(calc_price, theta, df_1102)
        ql_iv = iv_from_prices(ql_price, df_1102)
        scenarios[scenario] = {
            "ddn_iv_mre": calc_mre(ddn_iv, iv_true),
            "ql_iv_mre": calc_mre(ql_iv, iv_true),
            "ql_price_mre": calc_mre(ql_price, price_true),
            "ddn_iv_stats": error_stats(ddn_iv, iv_true),
            "ql_iv_stats": error_stats(ql_iv, iv_true),
            "ql_price_stats": error_stats(ql_price, price_true),
            "ddn_iv": [float(x) for x in ddn_iv],
            "ql_iv": [float(x) for x in ql_iv],
            "ql_price": [float(x) for x in ql_price],
        }

    return {
        "model": "Heston_Project",
        "objective": "Vega-weighted IV MSE + Feller penalty",
        "theta_2021_11_01": [float(x) for x in theta_1101],
        "theta_2021_11_02": [float(x) for x in theta_1102],
        "loss_2021_11_01": float(loss_1101),
        "loss_2021_11_02": float(loss_1102),
        "time_2021_11_01_sec": float(time_1101),
        "time_2021_11_02_sec": float(time_1102),
        "feller_2021_11_01": bool(project["check_feller"](float(theta_1101[0]), float(theta_1101[1]), float(theta_1101[2]))),
        "feller_2021_11_02": bool(project["check_feller"](float(theta_1102[0]), float(theta_1102[1]), float(theta_1102[2]))),
        "scenarios": scenarios,
    }


def run_zhang_realize(df_1101: pd.DataFrame, df_1102: pd.DataFrame) -> dict:
    project = load_project_model(ROOT / "Zhang_realize", default_input_dim=9)
    model, device = project["model"], project["device"]
    calibrator = project["calibrator_cls"](model, device, seed=42)

    t0 = time.time()
    theta_1101, loss_1101, _ = calibrator.calibrate(
        market_dict_zhang(df_1101),
        n_starts=10,
        lr=5e-3,
        max_steps=300,
        patience=50,
        verbose=False,
    )
    time_1101 = time.time() - t0

    t0 = time.time()
    theta_1102, loss_1102, _ = calibrator.calibrate(
        market_dict_zhang(df_1102),
        n_starts=10,
        lr=5e-3,
        max_steps=300,
        patience=50,
        verbose=False,
    )
    time_1102 = time.time() - t0

    iv_true = df_1102["iv_market"].values
    price_true = df_1102["P_mkt"].values
    calc_price = project["calculate_heston_price"]
    scenarios = {}
    for scenario, theta in [("cross_day", theta_1101), ("same_day", theta_1102)]:
        ddn_price = ddn_price_batch(model, device, theta, df_1102)
        ddn_iv = iv_from_prices(ddn_price, df_1102)
        ql_price = ql_price_batch(calc_price, theta, df_1102)
        ql_iv = iv_from_prices(ql_price, df_1102)
        scenarios[scenario] = {
            "ddn_price_mre": calc_mre(ddn_price, price_true),
            "ddn_iv_mre": calc_mre(ddn_iv, iv_true),
            "ql_price_mre": calc_mre(ql_price, price_true),
            "ql_iv_mre": calc_mre(ql_iv, iv_true),
            "ddn_price_stats": error_stats(ddn_price, price_true),
            "ddn_iv_stats": error_stats(ddn_iv, iv_true),
            "ql_price_stats": error_stats(ql_price, price_true),
            "ql_iv_stats": error_stats(ql_iv, iv_true),
            "ddn_price": [float(x) for x in ddn_price],
            "ddn_iv": [float(x) for x in ddn_iv],
            "ql_price": [float(x) for x in ql_price],
            "ql_iv": [float(x) for x in ql_iv],
        }

    return {
        "model": "Zhang_realize",
        "objective": "Normalised price MSE",
        "theta_2021_11_01": [float(x) for x in theta_1101],
        "theta_2021_11_02": [float(x) for x in theta_1102],
        "loss_2021_11_01": float(loss_1101),
        "loss_2021_11_02": float(loss_1102),
        "time_2021_11_01_sec": float(time_1101),
        "time_2021_11_02_sec": float(time_1102),
        "feller_2021_11_01": bool(2.0 * float(theta_1101[0]) * float(theta_1101[1]) > float(theta_1101[2]) ** 2),
        "feller_2021_11_02": bool(2.0 * float(theta_1102[0]) * float(theta_1102[1]) > float(theta_1102[2]) ** 2),
        "scenarios": scenarios,
    }


def build_data_summary(df_1101: pd.DataFrame, df_1102: pd.DataFrame) -> list[dict]:
    rows = []
    for date, df in [(DATES[0], df_1101), (DATES[1], df_1102)]:
        rows.append(
            {
                "date": date,
                "contracts": int(len(df)),
                "S0": float(df["S0"].iloc[0]),
                "DTE_min": float(df["DTE"].min()),
                "DTE_max": float(df["DTE"].max()),
                "tau_min": float(df["tau"].min()),
                "tau_max": float(df["tau"].max()),
                "K_min": float(df["K"].min()),
                "K_max": float(df["K"].max()),
                "log_moneyness_min": float(df["log_K_S0"].min()),
                "log_moneyness_max": float(df["log_K_S0"].max()),
                "P_mkt_min": float(df["P_mkt"].min()),
                "P_mkt_max": float(df["P_mkt"].max()),
                "iv_min": float(df["iv_market"].min()),
                "iv_max": float(df["iv_market"].max()),
                "vega_min": float(df["vega"].min()),
                "vega_max": float(df["vega"].max()),
                "r_min": float(df["r"].min()),
                "r_max": float(df["r"].max()),
            }
        )
    return rows


def write_result_csvs(results: list[dict], data_summary: list[dict]) -> None:
    pd.DataFrame(data_summary).to_csv(OUT_DIR / "nvda_data_summary.csv", index=False)

    param_names = ["kappa", "lambda", "sigma", "rho", "v0"]
    param_rows = []
    for res in results:
        for date_key in ["2021_11_01", "2021_11_02"]:
            row = {
                "model": res["model"],
                "date": date_key.replace("_", "-"),
                "loss": res[f"loss_{date_key}"],
                "time_sec": res[f"time_{date_key}_sec"],
                "feller": res[f"feller_{date_key}"],
            }
            row.update({name: value for name, value in zip(param_names, res[f"theta_{date_key}"])})
            param_rows.append(row)
    pd.DataFrame(param_rows).to_csv(OUT_DIR / "nvda_parameters.csv", index=False)

    metric_rows = []
    for res in results:
        for scenario, vals in res["scenarios"].items():
            row = {"model": res["model"], "scenario": scenario}
            for key, value in vals.items():
                if key.endswith("_mre"):
                    row[key] = value
            metric_rows.append(row)
    pd.DataFrame(metric_rows).to_csv(OUT_DIR / "nvda_calibration_summary.csv", index=False)


def write_validation_details(results: list[dict], df_target: pd.DataFrame) -> None:
    rows = []
    base_cols = ["DTE", "tau", "S0", "K", "P_mkt", "iv_market", "r", "log_K_S0"]
    for res in results:
        for scenario, vals in res["scenarios"].items():
            for i, row in df_target.reset_index(drop=True).iterrows():
                item = {col: float(row[col]) for col in base_cols}
                item.update({"model": res["model"], "scenario": scenario, "contract_index": int(i + 1)})
                if "ddn_iv" in vals:
                    item["ddn_iv"] = float(vals["ddn_iv"][i])
                    item["ddn_iv_rel_err"] = abs(item["ddn_iv"] - item["iv_market"]) / (abs(item["iv_market"]) + 1e-12)
                if "ddn_price" in vals:
                    item["ddn_price"] = float(vals["ddn_price"][i])
                    item["ddn_price_rel_err"] = abs(item["ddn_price"] - item["P_mkt"]) / (abs(item["P_mkt"]) + 1e-12)
                if "ql_iv" in vals:
                    item["ql_iv"] = float(vals["ql_iv"][i])
                    item["ql_iv_rel_err"] = abs(item["ql_iv"] - item["iv_market"]) / (abs(item["iv_market"]) + 1e-12)
                if "ql_price" in vals:
                    item["ql_price"] = float(vals["ql_price"][i])
                    item["ql_price_rel_err"] = abs(item["ql_price"] - item["P_mkt"]) / (abs(item["P_mkt"]) + 1e-12)
                rows.append(item)
    pd.DataFrame(rows).to_csv(OUT_DIR / "nvda_validation_details.csv", index=False)


def write_latex_tables(results: list[dict], data_summary: list[dict]) -> None:
    d1, d2 = data_summary
    data_tex = f"""% ============================================================
% Table: NVDA Market Data Description
% ============================================================
\\begin{{table}}[htbp]
  \\centering
  \\caption{{Description of NVDA Call Option Data Used for Real-Market Calibration}}
  \\label{{tab:nvda_data}}
  \\begin{{tabular}}{{lcc}}
    \\toprule
    \\textbf{{Property}} & \\textbf{{2021-11-01 (Calibration)}} & \\textbf{{2021-11-02 (Validation)}} \\\\
    \\midrule
    \\multicolumn{{3}}{{l}}{{\\textit{{Underlying Asset}}}} \\\\
    \\quad Spot price $S_0$ (\\$)       & {d1['S0']:.2f} & {d2['S0']:.2f} \\\\
    \\quad Days to expiry (DTE)        & [{d1['DTE_min']:.0f}, {d1['DTE_max']:.0f}] & [{d2['DTE_min']:.0f}, {d2['DTE_max']:.0f}] \\\\
    \\quad Time to maturity $\\tau$ (yr)& [{d1['tau_min']:.4f}, {d1['tau_max']:.4f}] & [{d2['tau_min']:.4f}, {d2['tau_max']:.4f}] \\\\
    \\midrule
    \\multicolumn{{3}}{{l}}{{\\textit{{Option Chain (after cleaning)}}}} \\\\
    \\quad Number of valid contracts   & {d1['contracts']} & {d2['contracts']} \\\\
    \\quad Strike range $K$ (\\$)       & [{d1['K_min']:.2f}, {d1['K_max']:.2f}] & [{d2['K_min']:.2f}, {d2['K_max']:.2f}] \\\\
    \\quad Log-moneyness $\\ln(K/S_0)$  & [{d1['log_moneyness_min']:+.4f}, {d1['log_moneyness_max']:+.4f}] & [{d2['log_moneyness_min']:+.4f}, {d2['log_moneyness_max']:+.4f}] \\\\
    \\quad Market IV range             & [{d1['iv_min']:.4f}, {d1['iv_max']:.4f}] & [{d2['iv_min']:.4f}, {d2['iv_max']:.4f}] \\\\
    \\quad Vega range                  & [{d1['vega_min']:.4f}, {d1['vega_max']:.4f}] & [{d2['vega_min']:.4f}, {d2['vega_max']:.4f}] \\\\
    \\midrule
    \\multicolumn{{3}}{{l}}{{\\textit{{Risk-Free Rate (continuous, interpolated)}}}} \\\\
    \\quad Matched $r$ range           & [{d1['r_min']:.6f}, {d1['r_max']:.6f}] & [{d2['r_min']:.6f}, {d2['r_max']:.6f}] \\\\
    \\quad Rate source                 & \\multicolumn{{2}}{{c}}{{US Par Yield Curve (Federal Reserve, 2020--2023)}} \\\\
    \\quad Interpolation method        & \\multicolumn{{2}}{{c}}{{Cubic spline on 1mo/2mo/3mo/6mo/1yr nodes}} \\\\
    \\midrule
    \\multicolumn{{3}}{{l}}{{\\textit{{Cleaning Criteria}}}} \\\\
    \\quad Option type                 & \\multicolumn{{2}}{{c}}{{Call only}} \\\\
    \\quad DTE filter                  & \\multicolumn{{2}}{{c}}{{40--300 days}} \\\\
    \\quad Log-moneyness filter        & \\multicolumn{{2}}{{c}}{{$\\ln(K/S_0) \\in [-0.15,\\;0.20]$}} \\\\
    \\quad Volume filter               & \\multicolumn{{2}}{{c}}{{Positive call volume required}} \\\\
    \\quad Final selection             & \\multicolumn{{2}}{{c}}{{10 nearest-the-money contracts, then lowest relative spread}} \\\\
    \\bottomrule
  \\end{{tabular}}
\\end{{table}}
"""
    (OUT_DIR / "table_nvda_data.tex").write_text(data_tex)

    heston = next(res for res in results if res["model"] == "Heston_Project")
    zhang = next(res for res in results if res["model"] == "Zhang_realize")

    def metric(res: dict, scenario: str, key: str) -> float:
        return res["scenarios"][scenario].get(key, float("nan"))

    calibration_tex = f"""% ============================================================
% Table: NVDA Same-Day and Cross-Day Calibration
% ============================================================
\\begin{{table}}[htbp]
  \\centering
  \\caption{{Same-Day and Cross-Day Calibration Performance on NVDA Options (2021-11-01 $\\rightarrow$ 2021-11-02, $M={N_CONTRACTS}$ contracts)}}
  \\label{{tab:nvda_calibration}}
  \\setlength{{\\tabcolsep}}{{6pt}}
  \\begin{{tabular}}{{llcc}}
    \\toprule
    \\textbf{{Scenario}} & \\textbf{{Metric}} & \\textbf{{Heston Project}} & \\textbf{{Zhang (2025)}} \\\\
    \\midrule
    \\multicolumn{{4}}{{l}}{{\\textit{{Same-Day: parameters calibrated on 2021-11-02 and evaluated on 2021-11-02}}}} \\\\
    \\quad Same-day & Native model IV MRE & {pct_tex(metric(heston, 'same_day', 'ddn_iv_mre'))} & {pct_tex(metric(zhang, 'same_day', 'ddn_iv_mre'))} \\\\
    \\quad Same-day & Native model price MRE & n/a & {pct_tex(metric(zhang, 'same_day', 'ddn_price_mre'))} \\\\
    \\quad Same-day & QuantLib IV MRE & {pct_tex(metric(heston, 'same_day', 'ql_iv_mre'))} & {pct_tex(metric(zhang, 'same_day', 'ql_iv_mre'))} \\\\
    \\quad Same-day & QuantLib price MRE & {pct_tex(metric(heston, 'same_day', 'ql_price_mre'))} & {pct_tex(metric(zhang, 'same_day', 'ql_price_mre'))} \\\\
    \\midrule
    \\multicolumn{{4}}{{l}}{{\\textit{{Cross-Day: parameters calibrated on 2021-11-01 and evaluated on 2021-11-02}}}} \\\\
    \\quad Cross-day & Native model IV MRE & {pct_tex(metric(heston, 'cross_day', 'ddn_iv_mre'))} & {pct_tex(metric(zhang, 'cross_day', 'ddn_iv_mre'))} \\\\
    \\quad Cross-day & Native model price MRE & n/a & {pct_tex(metric(zhang, 'cross_day', 'ddn_price_mre'))} \\\\
    \\quad Cross-day & QuantLib IV MRE & {pct_tex(metric(heston, 'cross_day', 'ql_iv_mre'))} & {pct_tex(metric(zhang, 'cross_day', 'ql_iv_mre'))} \\\\
    \\quad Cross-day & QuantLib price MRE & {pct_tex(metric(heston, 'cross_day', 'ql_price_mre'))} & {pct_tex(metric(zhang, 'cross_day', 'ql_price_mre'))} \\\\
    \\midrule
    \\multicolumn{{4}}{{l}}{{\\textit{{Degradation from same-day to cross-day}}}} \\\\
    \\quad $\\Delta$ native IV MRE & Cross-day minus same-day & {100.0 * (metric(heston, 'cross_day', 'ddn_iv_mre') - metric(heston, 'same_day', 'ddn_iv_mre')):+.2f} pp & {100.0 * (metric(zhang, 'cross_day', 'ddn_iv_mre') - metric(zhang, 'same_day', 'ddn_iv_mre')):+.2f} pp \\\\
    \\bottomrule
  \\end{{tabular}}
  \\begin{{tablenotes}}
    \\small
    \\item \\textit{{Note:}} IV MRE and price MRE are mean relative errors. The Heston Project model is calibrated directly in implied-volatility space with Vega weighting and a Feller penalty. The Zhang baseline is calibrated in normalised price space; its native IV MRE is computed by converting DDN-predicted prices to Black--Scholes implied volatilities. QuantLib rows re-price the selected NVDA contracts with the calibrated Heston parameters.
  \\end{{tablenotes}}
\\end{{table}}
"""
    (OUT_DIR / "table_nvda_calibration.tex").write_text(calibration_tex)

    param_lines = [
        "% ============================================================",
        "% Table: NVDA Calibrated Parameters",
        "% ============================================================",
        "\\begin{table}[htbp]",
        "  \\centering",
        "  \\caption{Calibrated Heston Parameters on NVDA Options}",
        "  \\label{tab:nvda_parameters}",
        "  \\begin{tabular}{llrrrrr}",
        "    \\toprule",
        "    \\textbf{Model} & \\textbf{Calibration Date} & $\\kappa$ & $\\theta$ & $\\sigma$ & $\\rho$ & $v_0$ \\\\",
        "    \\midrule",
    ]
    for res in results:
        model_label = res["model"].replace("_", "\\_")
        for date_key, date_label in [("2021_11_01", "2021-11-01"), ("2021_11_02", "2021-11-02")]:
            theta = res[f"theta_{date_key}"]
            param_lines.append(f"    {model_label} & {date_label} & " + " & ".join(f"{v:.6f}" for v in theta) + " \\\\")
    param_lines.extend(["    \\bottomrule", "  \\end{tabular}", "\\end{table}", ""])
    (OUT_DIR / "table_nvda_parameters.tex").write_text("\n".join(param_lines))


yield_data = pd.read_csv(ROOT / "data" / "par-yield-curve-rates-2020-2023.csv")
df_nvda_1101 = load_and_clean_nvda(yield_data, "2021-11-01")
df_nvda_1102 = load_and_clean_nvda(yield_data, "2021-11-02")

df_nvda_1101.to_csv(OUT_DIR / "nvda_selected_2021-11-01.csv", index=False)
df_nvda_1102.to_csv(OUT_DIR / "nvda_selected_2021-11-02.csv", index=False)

print("Running Heston_Project NVDA calibration...")
heston_result = run_heston_project(df_nvda_1101, df_nvda_1102)
print("Running Zhang_realize NVDA calibration...")
zhang_result = run_zhang_realize(df_nvda_1101, df_nvda_1102)

results = [heston_result, zhang_result]
data_summary = build_data_summary(df_nvda_1101, df_nvda_1102)
write_result_csvs(results, data_summary)
write_validation_details(results, df_nvda_1102)
write_latex_tables(results, data_summary)

(OUT_DIR / "nvda_calibration_results.json").write_text(json.dumps({"data_summary": data_summary, "results": results}, indent=2))

summary_df = pd.read_csv(OUT_DIR / "nvda_calibration_summary.csv")
print("\nNVDA calibration summary:")
print(summary_df.to_string(index=False))
print("\nWrote LaTeX tables:")
print(OUT_DIR / "table_nvda_data.tex")
print(OUT_DIR / "table_nvda_calibration.tex")
print(OUT_DIR / "table_nvda_parameters.tex")
